In [2]:
# 1. 라이브러리
import os, shutil
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

BASE_PATH = "/kaggle/input/competitions/global-wheat-detection"
WORK_PATH = "/kaggle/working/wheat_yolo"

In [3]:
# 2. 데이터 로드 + bbox 파싱
df = pd.read_csv(f"{BASE_PATH}/train.csv")

df["bbox"] = df["bbox"].apply(lambda x: np.array(x[1:-1].split(","), dtype=float))
df[["x", "y", "w", "h"]] = pd.DataFrame(df["bbox"].tolist(), index=df.index)

image_ids = df["image_id"].unique()

train_ids, val_ids = train_test_split(
    image_ids,
    test_size=0.2,
    random_state=42
)

In [4]:
# 3. YOLO 폴더 생성
for split in ["train", "val"]:
    os.makedirs(f"{WORK_PATH}/images/{split}", exist_ok=True)
    os.makedirs(f"{WORK_PATH}/labels/{split}", exist_ok=True)

In [5]:
# 4. YOLO 형식으로 변환
def make_yolo_data(ids, split):
    for img_id in ids:
        shutil.copy(
            f"{BASE_PATH}/train/{img_id}.jpg",
            f"{WORK_PATH}/images/{split}/{img_id}.jpg"
        )
        
        rows = df[df["image_id"] == img_id]
        
        with open(f"{WORK_PATH}/labels/{split}/{img_id}.txt", "w") as f:
            for _, row in rows.iterrows():
                x, y, w, h = row["x"], row["y"], row["w"], row["h"]
                img_w, img_h = row["width"], row["height"]
                
                x_center = (x + w / 2) / img_w
                y_center = (y + h / 2) / img_h
                w = w / img_w
                h = h / img_h
                
                f.write(f"0 {x_center} {y_center} {w} {h}\n")

make_yolo_data(train_ids, "train")
make_yolo_data(val_ids, "val")

KeyboardInterrupt: 

In [ ]:
# 5. data.yaml 생성
yaml_text = f"""
train: {WORK_PATH}/images/train
val: {WORK_PATH}/images/val

nc: 1
names: ['wheat']
"""

with open(f"{WORK_PATH}/data.yaml", "w") as f:
    f.write(yaml_text)

In [ ]:
# YOLOv8 설치
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")

model.train(
    data="/kaggle/working/wheat_yolo/data.yaml",
    imgsz=640,
    batch=16,
    epochs=20,

    # augmentation
    mosaic=1.0,
    mixup=0.1,
    scale=0.5,

    multi_scale=True,
    cos_lr=True,
    label_smoothing=0.05,

    name="wheat_improved_v8m"
)

In [ ]:
metrics = model.val(
    data="/kaggle/working/wheat_yolo/data.yaml",
    imgsz=640,
    augment=True
    conf=0.001
)

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)